# Fit a plan to your budget

> Get exactly the number of samples that you can afford

::: {.callout-note appearance="simple" icon=false}
**How-to guide: get one job done.** It assumes that you know the basics. New to trufl? Take the [tutorial](../tutorials/01_getting_started.ipynb) first. [About the four kinds of page](../explanation/06_documentation.ipynb).
:::

**You need:** `areas`, and their `ranks` from `Optimizer.get_rank`.

`rank_to_sample` [does not always return the budget you ask for](../explanation/04_allocation.ipynb), so use these steps.

In [ ]:
#| hide
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import geopandas as gpd
from fastcore.test import test_eq

from trufl.utils import gridder
from trufl.callbacks import State, MaxCB, MinCB, StdCB, CountCB, PriorCB
from trufl.optimizer import Optimizer
from trufl.sampler import Sampler, rank_to_sample
from trufl.collector import DataCollector

fname_raster = '../files/ground-truth-01-4326-simulated.tif'
areas = gridder(fname_raster)
no_samples = gpd.GeoDataFrame(index=pd.Index([], name='loc_id'), geometry=None, data={'value': None})
prior_state = State(no_samples, areas, cbs=[PriorCB(fname_raster)])()
first_ranks = Optimizer(prior_state).get_rank(is_benefit_x=[True], w_vector=[1], s_method='CP')
first_n = rank_to_sample(first_ranks['rank'].reindex(areas.index).to_numpy(), budget=600, min=1, policy='quantiles')
measurements = DataCollector(fname_raster).collect(
    Sampler(areas).sample(first_n, method='uniform', rng=np.random.default_rng(0)))
state = State(measurements, areas, cbs=[MaxCB(), MinCB(), StdCB(), CountCB(), PriorCB(fname_raster)])()
ranks = Optimizer(state).get_rank(is_benefit_x=[True, True, True, False, True],
                                  w_vector=[0.25, 0.125, 0.125, 0.25, 0.25], n_method='LINEAR1', s_method='CP')

## 1. Put the ranks in the order of the areas

In [ ]:
r = ranks['rank'].reindex(areas.index)
assert r.notna().all(), 'some areas have no rank'
r = r.to_numpy()

## 2. Set the budget and check that it covers the minimum

In [ ]:
budget, minimum, policy = 130, 1, 'quantiles'
assert budget >= minimum * len(areas), 'the budget cannot pay for the minimum: see the last section'

## 3. Find the request that comes closest without exceeding it

In [ ]:
import numpy as np
from trufl.sampler import rank_to_sample

totals = {b: rank_to_sample(r, budget=b, min=minimum, policy=policy).sum() for b in range(1, 2 * budget + 1)}
request = max((b for b, total in totals.items() if total <= budget), key=totals.get)
n = rank_to_sample(r, budget=request, min=minimum, policy=policy)
print(n.sum(), 'of', budget)

125 of 130


## 4. Give what is left to the top of the list

In [ ]:
spare = budget - n.sum()
n += spare // len(n)
n[np.argsort(r)[:spare % len(n)]] += 1

assert n.sum() == budget and n.min() >= minimum
print(n.sum(), 'of', budget)

130 of 130


Keep `n` as it is: its order is that of `areas`.

## If the budget cannot pay for the minimum

Either set `minimum = 0` and repeat from step 2, or send one sample to each of the first areas of the list until the budget is used:

In [ ]:
small_budget = 20
n_small = (r <= small_budget).astype(int)
print(n_small.sum(), 'of', small_budget)

20 of 20


In [ ]:
#| hide
test_eq([n.sum(), n_small.sum()], [130, 20])

Next: [keep teams out of closed zones](05_closed_zones.ipynb), or [give the teams their points](06_field_list.ipynb).